In [ ]:
from src.data.nonogram_dataset import NonogramDataset
from src.models.simple_nn import SimpleNeuralNetwork
from src.training.trainer import Trainer
from torch.utils.data import DataLoader, random_split
import torch


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

dataset = NonogramDataset(
    "../data/processed/train_small.npz", 
    "../data/processed/target_small.npz", 
    nonogram_size=5,
    flat=True)

model = SimpleNeuralNetwork(
    input_size=dataset.flat_input_size, 
    hidden_size=256, 
    output_size=dataset.flat_target_size,
    num_layers=5).to(device)

In [ ]:
train_dataset, test_dataset = random_split(
    dataset, 
    [int(0.8 * len(dataset)), int(0.2 * len(dataset))],
    generator=torch.Generator().manual_seed(42)
)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
trainer = Trainer(model, train_loader, test_loader, criterion, optimizer, device, epochs=20)

In [ ]:
trainer.train()

In [ ]:
print(f"Model Accuracy: {trainer.test_accuracy():.4f}")